# TalkToTheCell — SmolVLA fine-tune on the sim2cell datasets

Fine-tunes **SmolVLA (450M)** — a vision-language-action model — on one of the
author's language-labeled MuJoCo pick-and-place datasets
(*"Pick up the red block and place it in the blue tray."*):

- `so101-sim-pickplace` (100 eps, nominal) — the original run: **0% → 55%**
- `so101-sim-pickplace-v2` (160 eps, incl. recovery demos) — the coverage fix:
  the 55% model's failures were all non-engagements on right-side spawns

The trained policy is pushed to your Hub account automatically.

**Kaggle setup (required before running):**
1. Accelerator: **GPU T4 x2** — Settings panel on the right
2. Internet: **ON** (Settings → Internet)
3. Secret: **Add-ons → Secrets → `HF_TOKEN`** = your Hugging Face *write* token

Measured 2.43 s/step on a T4 → 12k steps ≈ 8.1 h (fits the 12 h session cap;
20k would not). Defaults freeze the VLM and train only the action expert, so
the 450M model fits a 16 GB T4 with AMP. Checkpoints save every 5k steps.

In [ ]:
# ---- parameters: flip these per run ----
DATASET = "ahmedsohail2003/so101-sim-pickplace-v2"
MODEL_REPO = "ahmedsohail2003/smolvla-so101-pickplace-v2"
STEPS = 12000
print(DATASET, "->", MODEL_REPO, f"({STEPS} steps)")

In [ ]:
# ~3-4 min: LeRobot 0.6.0 (same version the dataset was recorded with).
# Extras: smolvla (VLA deps) + dataset (av video decoding — required by lerobot.datasets)
!pip install -q "lerobot[smolvla,dataset]==0.6.0"
!nvidia-smi

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

login(UserSecretsClient().get_secret("HF_TOKEN"))
print("logged in as:", whoami()["name"])

In [ ]:
# Sanity: the dataset loads from the Hub (metadata only, fast)
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata(DATASET)
print(meta.total_episodes, "episodes,", meta.total_frames, "frames,", meta.fps, "fps")
print("cameras:", list(meta.camera_keys))

In [ ]:
# The fine-tune. Measured 2.43 s/step on a T4 -> 12k steps ~= 8.1 h (fits the
# 12 h Kaggle session cap with margin).
# - policy.path pulls the pretrained SmolVLA base; defaults freeze the VLM and
#   train only the action expert
# - rename_map: the base expects cameras named camera1/2/3; ours are front/wrist.
#   Renamed, our 2 cams are an accepted subset; the absent camera3 is skipped.
# - the command is built as a Python string because IPython's ! interpolation
#   can't mix {var} placeholders with the rename_map's literal JSON braces
RENAME = '{"observation.images.front": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'
cmd = (
    "lerobot-train"
    " --policy.path=lerobot/smolvla_base"
    f" --dataset.repo_id={DATASET}"
    f" --rename_map='{RENAME}'"
    " --batch_size=8"
    f" --steps={STEPS}"
    " --save_freq=5000"
    " --log_freq=100"
    " --num_workers=2"
    " --policy.use_amp=true"
    " --policy.device=cuda"
    " --policy.push_to_hub=true"
    f" --policy.repo_id={MODEL_REPO}"
    " --output_dir=/kaggle/working/train_smolvla"
    " --wandb.enable=false"
)
print(cmd, "\n")
!{cmd}

In [ ]:
# Confirm the model landed on the Hub
from huggingface_hub import HfApi

files = HfApi().list_repo_files(MODEL_REPO)
print("\n".join(files))
print(f"\nhttps://huggingface.co/{MODEL_REPO}")

## If the session dies mid-run

Checkpoints are in `/kaggle/working/train_smolvla/checkpoints/`. Save the run's
output as a Kaggle dataset (File → Save Version keeps `/kaggle/working`), then
in a fresh session re-run the install/login cells and resume with:

```
!lerobot-train --config_path=<checkpoint_dir>/pretrained_model/train_config.json --resume=true
```

## After training (back on the local machine)

Evaluate in the MuJoCo work-cell against the ACT baseline (65% / 75% w/ ensembling)
and record the before/after demo video — SmolVLA inference fits the local RTX 4050.